# 01 · Train ResNet-18 on CIFAR-10 (with intermediate checkpoints)

Trains a CIFAR-style ResNet-18 from scratch for 350 epochs and saves model checkpoints at 113 selected epochs.
These checkpoints are the input to the two analysis notebooks:

- `02_compute_cka.ipynb` – layer-wise Centered Kernel Alignment (CKA) against the final model
- `03_compute_neural_collapse.ipynb` – Neural Collapse metrics (NC1–NC4)

**Setup**

| Item | Value |
|---|---|
| Model | torchvision `resnet18`, `conv1` → 3×3 / stride 1 / pad 1, `maxpool` removed, 10 outputs |
| Data | CIFAR-10 (50k train / 10k test), random crop (pad 4) + horizontal flip, per-channel normalisation |
| Optimiser | SGD, lr 0.1, momentum 0.9, weight decay 5e-4, batch size 128 |
| Schedule | Cosine annealing, `T_max=350`, `eta_min=1e-5` |
| Epochs | 350 |
| Seed | 42 |

**Outputs** (written to `OUTPUT_DIR`)

- `checkpoints/epoch_{N}.pth` – dict with `epoch` and `model_state_dict`, for the 113 epochs in `SAVE_EPOCHS`
- `training_history.csv` – per-epoch `train_loss`, `train_acc`, `test_acc`
- `final_model.pth` – final `state_dict`

> **Note:** `train_loss` / `train_acc` are running averages over the *augmented* training batches in train mode, not a separate clean pass over the training set.

In [ ]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

## Configuration

In [ ]:
# Paths: use /kaggle/working on Kaggle, the current directory elsewhere.
ON_KAGGLE = os.path.exists("/kaggle/working")
BASE_DIR = "/kaggle/working" if ON_KAGGLE else "."

DATA_DIR = os.path.join(BASE_DIR, "data")            # CIFAR-10 is downloaded here if missing
OUTPUT_DIR = BASE_DIR
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Hyperparameters
SEED = 42
NUM_EPOCHS = 350
BATCH_SIZE = 128
LR = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
ETA_MIN = 1e-5
NUM_WORKERS = 2

# Epochs at which a checkpoint is saved (113 in total)
SAVE_EPOCHS = sorted(set(
    list(range(1, 21))
    + list(range(25, 101, 5))
    + list(range(110, 221, 10))
    + list(range(220, 351, 2))
))
print(f"{len(SAVE_EPOCHS)} checkpoints will be saved")

## Reproducibility and device

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Data

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2023, 0.1994, 0.2010)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

train_dataset = datasets.CIFAR10(root=DATA_DIR, train=True, download=True, transform=transform_train)
test_dataset = datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

## Model, loss, optimiser, scheduler

In [ ]:
model = models.resnet18(num_classes=10)
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=ETA_MIN)

## Train / evaluate functions

In [ ]:
@torch.no_grad()
def evaluate():
    model.eval()
    correct, total = 0, 0
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return 100 * correct / total


def train_one_epoch():
    model.train()
    running_loss, correct, total = 0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return running_loss / len(train_loader), 100 * correct / total

## Training loop

In [ ]:
history = {"epoch": [], "train_loss": [], "train_acc": [], "test_acc": []}

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch()
    test_acc = evaluate()
    scheduler.step()

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_acc"].append(test_acc)

    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch:03d} | LR {current_lr:.6f} | Loss {train_loss:.4f} | "
          f"Train {train_acc:.2f}% | Test {test_acc:.2f}%")

    if epoch in SAVE_EPOCHS:
        torch.save(
            {"epoch": epoch, "model_state_dict": model.state_dict()},
            os.path.join(CHECKPOINT_DIR, f"epoch_{epoch}.pth"),
        )

## Save history and final model

In [ ]:
history_df = pd.DataFrame(history)
history_df.to_csv(os.path.join(OUTPUT_DIR, "training_history.csv"), index=False)

torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "final_model.pth"))